<a href="https://colab.research.google.com/github/bayanasar/membraneclaw/blob/main/experiments/notebooks/04_reinforce_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — REINFORCE, Baselines, and a Critic

Everything so far has been about **evaluation**: given a policy, how good is it?
Notebook 02 improved policies indirectly — compute values, then read off the
greedy action. That works when you can compute values for every state.

This notebook takes the other route. Instead of learning values and deriving a
policy, we **parameterise the policy directly** and push its parameters uphill
on expected return. That is a *policy gradient* method, and its simplest form is
**REINFORCE**.

Then we confront the thing that makes REINFORCE painful in practice — variance —
and fix it twice: with a **baseline**, and with a learned **critic**.

As before, `step()` is the only environment access. Notebook 02's exact answers
serve purely as the grading key.

In [ ]:
# --- environment from notebook 01, repeated so this notebook stands alone ---
from __future__ import annotations

from enum import IntEnum
from typing import NamedTuple

import numpy as np

class State(IntEnum):
    NO_INFO = 0           # nothing checked yet
    SALINITY_CHECKED = 1  # feed salinity known
    FOULING_CHECKED = 2   # fouling indicators known
    BOTH_CHECKED = 3      # both kinds of evidence in hand
    SUCCESS = 4           # problem solved (terminal)
    FAILURE = 5           # wrong or unsafe fix submitted (terminal)


class Action(IntEnum):
    CHECK_SALINITY = 0
    CHECK_FOULING = 1
    RUN_SIMULATION = 2
    SUBMIT_DIRECTLY = 3


N_STATES, N_ACTIONS = len(State), len(Action)
TERMINAL_STATES = frozenset({State.SUCCESS, State.FAILURE})
NONTERMINAL = [s for s in State if s not in TERMINAL_STATES]


def is_terminal(state) -> bool:
    return State(state) in TERMINAL_STATES


COST_CHECK = -0.5          # first look at a piece of evidence
COST_REPEAT_CHECK = -1.0   # re-checking something already known: pure waste
REWARD_SIM_SUCCESS = 9.0   # +10 outcome, minus the -1 implicit cost of simulating
REWARD_SIM_FAILURE = -11.0
REWARD_SUBMIT_SUCCESS = 10.0
REWARD_SUBMIT_FAILURE = -10.0

SIM_SUCCESS_PROB = {
    State.NO_INFO: 0.15,
    State.SALINITY_CHECKED: 0.55,
    State.FOULING_CHECKED: 0.45,
    State.BOTH_CHECKED: 0.95,
}
SUBMIT_SUCCESS_PROB = {
    State.NO_INFO: 0.05,
    State.SALINITY_CHECKED: 0.35,
    State.FOULING_CHECKED: 0.25,
    State.BOTH_CHECKED: 0.75,
}


class Transition(NamedTuple):
    prob: float
    next_state: State
    reward: float


# Evidence held in each non-terminal state, used to work out where a check lands.
_EVIDENCE = {
    State.NO_INFO: frozenset(),
    State.SALINITY_CHECKED: frozenset({Action.CHECK_SALINITY}),
    State.FOULING_CHECKED: frozenset({Action.CHECK_FOULING}),
    State.BOTH_CHECKED: frozenset({Action.CHECK_SALINITY, Action.CHECK_FOULING}),
}
_STATE_BY_EVIDENCE = {ev: st for st, ev in _EVIDENCE.items()}


def transitions(state, action) -> tuple[Transition, ...]:
    # Every outcome of taking `action` in `state`, probabilities summing to 1.
    state, action = State(state), Action(action)

    if is_terminal(state):
        return (Transition(1.0, state, 0.0),)

    if action in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
        already_known = action in _EVIDENCE[state]
        next_state = (
            state if already_known
            else _STATE_BY_EVIDENCE[_EVIDENCE[state] | {action}]
        )
        reward = COST_REPEAT_CHECK if already_known else COST_CHECK
        return (Transition(1.0, next_state, reward),)

    if action is Action.RUN_SIMULATION:
        p = SIM_SUCCESS_PROB[state]
        return (
            Transition(p, State.SUCCESS, REWARD_SIM_SUCCESS),
            Transition(1.0 - p, State.FAILURE, REWARD_SIM_FAILURE),
        )

    p = SUBMIT_SUCCESS_PROB[state]
    return (
        Transition(p, State.SUCCESS, REWARD_SUBMIT_SUCCESS),
        Transition(1.0 - p, State.FAILURE, REWARD_SUBMIT_FAILURE),
    )

def transition_tables() -> tuple[np.ndarray, np.ndarray]:
    # Dense tables for exact methods: P[s, a, s'] and expected R[s, a].
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        for a in Action:
            for prob, next_state, reward in transitions(s, a):
                P[s, a, next_state] += prob
                R[s, a] += prob * reward
    return P, R


P, R = transition_tables()

def policy_matrices(pi, P, R):
    # Collapse an MDP + policy into an MRP: (P_pi [S,S], R_pi [S]).
    P_pi = np.einsum("sa,sat->st", pi, P)
    R_pi = np.einsum("sa,sa->s", pi, R)
    return P_pi, R_pi


def deterministic(action_of_state) -> np.ndarray:
    # Build a [S, A] policy matrix from a {state: action} mapping.
    pi = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        pi[s, action_of_state(s)] = 1.0
    return pi


def step(state, action, rng) -> tuple[State, float, bool]:
    # Sample one environment step: (next_state, reward, done).
    # Inverse-CDF sampling from a single uniform draw. The obvious
    # rng.choice(..., p=probs) is ~4x slower per call, which matters once the
    # later notebooks run hundreds of thousands of episodes.
    outcomes = transitions(state, action)
    if len(outcomes) == 1:
        only = outcomes[0]
        return only.next_state, only.reward, is_terminal(only.next_state)
    u, cumulative = rng.random(), 0.0
    for prob, next_state, reward in outcomes:
        cumulative += prob
        if u < cumulative:
            return next_state, reward, is_terminal(next_state)
    last = outcomes[-1]  # float-rounding fallback
    return last.next_state, last.reward, is_terminal(last.next_state)


def rollout(policy_fn, rng, max_steps=20):
    # Run one episode; return a list of (state, action, reward) triples.
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = policy_fn(s, rng)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r))
        s = ns
        if done:
            break
    return traj


def show(traj):
    total = sum(r for _, _, r in traj)
    for s, a, r in traj:
        print(f"  {State(s).name:<18} --{Action(a).name:<16}--> {r:+.1f}")
    print(f"  total (undiscounted) = {total:+.1f}")
    return total

GAMMA = 0.95

# Ground truth from notebook 02, for grading only - never used while learning.
V_STAR = np.array([6.245, 7.1, 7.1, 8.0, 0.0, 0.0])

# At NO_INFO both checks are tied at q* = 6.245 - order does not matter, only
# that both get done. So "optimal" there means "either check".
OPTIMAL_ACTIONS = {
    State.NO_INFO: {Action.CHECK_SALINITY, Action.CHECK_FOULING},
    State.SALINITY_CHECKED: {Action.CHECK_FOULING},
    State.FOULING_CHECKED: {Action.CHECK_SALINITY},
    State.BOTH_CHECKED: {Action.RUN_SIMULATION},
}
print("target: v*(NO_INFO) =", V_STAR[State.NO_INFO])

## A parameterised policy

We store one real number $\theta_{s,a}$ per state-action pair — the *preference*
for taking $a$ in $s$ — and convert preferences to probabilities with a softmax:

$$\pi_\theta(a \mid s) = \frac{e^{\theta_{s,a}}}{\sum_{a'} e^{\theta_{s,a'}}}$$

Softmax has three properties we want: every action keeps nonzero probability
(so exploration never fully dies), the policy is differentiable in $\theta$, and
it can approach determinism as preferences spread apart.

Starting from $\theta = 0$ gives a uniform random policy.

In [ ]:
def softmax(x, axis=-1):
    # Subtract the max before exponentiating: standard overflow guard.
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def policy_probs(theta):
    return softmax(theta, axis=1)          # [S, A]


theta0 = np.zeros((N_STATES, N_ACTIONS))
print("uniform policy at theta = 0:")
print(np.round(policy_probs(theta0)[State.NO_INFO], 4))

## The policy gradient theorem

We want to maximise $J(\theta) = \mathbb{E}_{\pi_\theta}[G_0]$. The obstacle is
that $\theta$ affects the *distribution* of trajectories, not the reward
function, so we cannot differentiate through the environment.

The policy gradient theorem sidesteps that:

$$\nabla_\theta J(\theta)
  = \mathbb{E}_{\pi_\theta}\Big[\sum_{t} G_t \, \nabla_\theta \log \pi_\theta(A_t \mid S_t)\Big]$$

Everything on the right is available from sampled experience: $G_t$ is the
return we computed in notebook 03, and $\nabla_\theta \log \pi_\theta$ is a
property of our own policy that we can differentiate by hand.

Read it as a weighted vote: $\nabla \log \pi$ points in the direction that makes
the action just taken **more likely**, and $G_t$ scales that push by how well
things went. Good returns reinforce; bad returns suppress. Hence the name.

For a softmax over tabular preferences, the gradient is famously clean — for
the state $s$ that was visited and the action $a$ that was taken:

$$\frac{\partial \log \pi_\theta(a \mid s)}{\partial \theta_{s,a'}}
  = \mathbb{1}[a' = a] - \pi_\theta(a' \mid s)$$

That is "one-hot minus the current probabilities": push up the chosen action,
push down everything in proportion to how likely it already was.

In [ ]:
def grad_log_pi(theta, s, a):
    # d log pi(a|s) / d theta[s, :]  =  onehot(a) - pi(.|s)
    g = np.zeros((N_STATES, N_ACTIONS))
    g[s] = -policy_probs(theta)[s]
    g[s, a] += 1.0
    return g


# Sanity check the analytic gradient against a finite-difference estimate.
rng = np.random.default_rng(0)
theta_test = rng.normal(size=(N_STATES, N_ACTIONS)) * 0.5
s_t, a_t, eps = State.SALINITY_CHECKED, Action.CHECK_FOULING, 1e-6

analytic = grad_log_pi(theta_test, s_t, a_t)
numeric = np.zeros_like(theta_test)
for a in Action:
    bumped = theta_test.copy()
    bumped[s_t, a] += eps
    numeric[s_t, a] = (
        np.log(policy_probs(bumped)[s_t, a_t]) - np.log(policy_probs(theta_test)[s_t, a_t])
    ) / eps

print("max |analytic - numeric| =", np.abs(analytic - numeric).max())
print("gradient is correct:", np.allclose(analytic, numeric, atol=1e-5))

Always do this check when hand-deriving a gradient. A subtly wrong gradient
still *trains* — it just converges to the wrong place, slowly, and looks like a
hyperparameter problem for a week.

## REINFORCE

The algorithm in full:

1. run an episode with the current policy
2. compute the return $G_t$ at every step
3. update $\theta \leftarrow \theta + \alpha \sum_t \gamma^t G_t \nabla_\theta \log \pi_\theta(A_t \mid S_t)$
4. repeat

The $\gamma^t$ factor is the theoretically correct discounting of later steps.
It is often dropped in practice; we keep it so the objective matches the $v_\pi$
computed in earlier notebooks.

In [ ]:
def sample_action(theta, s, rng):
    p = policy_probs(theta)[s]
    return Action(int(np.searchsorted(np.cumsum(p), rng.random())))


def run_episode(theta, rng, max_steps=20):
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = sample_action(theta, s, rng)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r))
        s = ns
        if done:
            break
    return traj


def returns_along(traj, gamma=GAMMA):
    G, out = 0.0, []
    for _, _, r in reversed(traj):
        G = r + gamma * G
        out.append(G)
    return list(reversed(out))


def reinforce(n_episodes, alpha, rng, gamma=GAMMA, seed_theta=None):
    # Vanilla REINFORCE. Returns (theta, learning curve of episode returns).
    theta = np.zeros((N_STATES, N_ACTIONS)) if seed_theta is None else seed_theta.copy()
    curve = np.zeros(n_episodes)

    for ep in range(n_episodes):
        traj = run_episode(theta, rng)
        Gs = returns_along(traj, gamma)
        grad = np.zeros_like(theta)
        for t, ((s, a, _), G) in enumerate(zip(traj, Gs)):
            grad += (gamma ** t) * G * grad_log_pi(theta, s, a)
        theta += alpha * grad
        curve[ep] = Gs[0]
    return theta, curve

### Training

In [ ]:
rng = np.random.default_rng(1)
theta_rf, curve_rf = reinforce(3_000, alpha=0.02, rng=rng)


def smooth(x, w=100):
    return np.convolve(x, np.ones(w) / w, mode="valid")


def n_optimal(theta):
    probs = policy_probs(theta)
    return sum(Action(probs[s].argmax()) in OPTIMAL_ACTIONS[s] for s in NONTERMINAL)


def report(theta, label):
    probs = policy_probs(theta)
    print(f"{label}:")
    print(f"  {'state':<18}{'best action':<17}{'p(best)':>9}   {'optimal?'}")
    for s in NONTERMINAL:
        best = Action(probs[s].argmax())
        ok = "yes" if best in OPTIMAL_ACTIONS[s] else "NO"
        print(f"  {s.name:<18}{best.name:<17}{probs[s].max():>9.3f}   {ok}")


report(theta_rf, "REINFORCE after 3,000 episodes")

sm = smooth(curve_rf)
print(f"\nmean return, first 100 episodes: {curve_rf[:100].mean():+.3f}")
print(f"mean return, last  100 episodes: {curve_rf[-100:].mean():+.3f}")
print(f"optimal (v* at NO_INFO):         {V_STAR[State.NO_INFO]:+.3f}")

REINFORCE recovers an optimal policy from **nothing but sampled returns** — no
model, no value function, no dynamic programming. It started uniform-random and
converged on the behaviour value iteration proved optimal in notebook 02.

Note which optimum it found: at `NO_INFO` it prefers `CHECK_FOULING`, whereas
value iteration's `argmax` picked `CHECK_SALINITY`. Both are correct — notebook
02 showed the two checks are tied at `q* = 6.245`. Gradient methods break ties
by whichever action happened to get reinforced first, so a run that looks
different from the dynamic-programming answer is not necessarily worse. Compare
*values*, not action labels.

The final return `+6.42` slightly exceeds `v*(NO_INFO) = 6.245` for a mundane
reason: it is a 100-episode sample average, so it carries sampling noise of a
few tenths. It is not evidence of beating the optimum, which is impossible.

Let us look at how the learning curve got there.

In [ ]:
print("learning curve (mean return over 100-episode windows):")
for frac in [0.0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]:
    i = min(int(frac * (len(sm) - 1)), len(sm) - 1)
    bar = "#" * max(0, int((sm[i] + 12) * 2))
    print(f"  ep {i:>5}  {sm[i]:>+7.3f}  {bar}")

## The variance problem

REINFORCE works, but watch what the update actually does. The gradient is scaled
by $G_t$ — the **raw return**. In this MDP nearly every return is either about
`+7` or about `-11`, so the updates are large in both directions and largely
cancel.

Worse, consider a state where *every* action gives a positive return. REINFORCE
increases the probability of whichever action was sampled, regardless of whether
it was the *best* one. It only sorts them out because better actions get sampled
with higher returns more often — a signal that has to fight its way through the
noise.

Let us measure that noise directly: estimate the gradient many times from the
same policy and look at how much the estimates disagree.

In [ ]:
def gradient_samples(theta, n, rng, gamma=GAMMA, baseline=None):
    # n independent single-episode gradient estimates, flattened.
    out = np.zeros((n, N_STATES * N_ACTIONS))
    for i in range(n):
        traj = run_episode(theta, rng)
        Gs = returns_along(traj, gamma)
        g = np.zeros((N_STATES, N_ACTIONS))
        for t, ((s, a, _), G) in enumerate(zip(traj, Gs)):
            adv = G if baseline is None else G - baseline[s]
            g += (gamma ** t) * adv * grad_log_pi(theta, s, a)
        out[i] = g.ravel()
    return out


theta_uniform = np.zeros((N_STATES, N_ACTIONS))   # where training actually starts

rng = np.random.default_rng(4)
g_plain = gradient_samples(theta_uniform, 4_000, rng)

print("gradient estimates from the uniform policy (4,000 single-episode samples)")
print(f"  mean gradient norm      : {np.linalg.norm(g_plain.mean(axis=0)):.4f}")
print(f"  mean per-sample variance: {g_plain.var(axis=0).mean():.4f}")
print(f"  signal-to-noise         : "
      f"{np.linalg.norm(g_plain.mean(axis=0)) / np.sqrt(g_plain.var(axis=0).mean()):.4f}")

The per-sample variance dwarfs the mean. Each individual episode points in a
wildly different direction, and only the *average* over many episodes points
uphill. That is why REINFORCE needs a small learning rate and many episodes:
it is averaging out enormous noise.

## Variance reduction with a baseline

Here is the key identity. For any function $b(s)$ that does not depend on the
action:

$$\mathbb{E}_{a \sim \pi}\big[b(s)\, \nabla_\theta \log \pi_\theta(a \mid s)\big] = 0$$

Because $\sum_a \pi(a|s) \nabla \log \pi(a|s) = \sum_a \nabla \pi(a|s) = \nabla \sum_a \pi(a|s) = \nabla 1 = 0$.

So we may subtract any such $b(s)$ from the return without changing the
gradient's expectation:

$$\nabla_\theta J = \mathbb{E}\Big[\sum_t \big(G_t - b(S_t)\big) \nabla_\theta \log \pi_\theta(A_t \mid S_t)\Big]$$

**Unbiased, but lower variance.** The best simple choice is $b(s) \approx
v_\pi(s)$, which turns the scale factor into the **advantage**
$G_t - v(S_t)$: "how much better was this than typical for this state?" Actions
that beat the state's average get reinforced; actions that fall short get
suppressed — even when both had positive returns.

Let us verify the variance claim empirically, using the true $v_\pi$ as baseline.

In [ ]:
def evaluate_exact(pi, gamma=GAMMA):
    P_pi, R_pi = policy_matrices(pi, P, R)
    return np.linalg.solve(np.eye(N_STATES) - gamma * P_pi, R_pi)


v_uniform = evaluate_exact(policy_probs(theta_uniform))   # exact baseline, for the experiment

rng = np.random.default_rng(4)                            # same seed as before
g_base = gradient_samples(theta_uniform, 4_000, rng, baseline=v_uniform)

mean_p, mean_b = g_plain.mean(axis=0), g_base.mean(axis=0)
var_p, var_b = g_plain.var(axis=0).mean(), g_base.var(axis=0).mean()

print("baseline v_pi(s) =", np.round(v_uniform[:4], 2), "\n")
print(f"{'':<22}{'no baseline':>14}{'with baseline':>15}")
print(f"{'mean gradient norm':<22}{np.linalg.norm(mean_p):>14.4f}{np.linalg.norm(mean_b):>15.4f}")
print(f"{'mean variance':<22}{var_p:>14.4f}{var_b:>15.4f}")
print(f"{'signal-to-noise':<22}{np.linalg.norm(mean_p)/np.sqrt(var_p):>14.4f}"
      f"{np.linalg.norm(mean_b)/np.sqrt(var_b):>15.4f}")
print(f"\nvariance reduced by {100 * (1 - var_b / var_p):.1f}%")
print(f"cosine between the two mean gradients: "
      f"{mean_p @ mean_b / (np.linalg.norm(mean_p) * np.linalg.norm(mean_b)):.4f}")

Two numbers matter here.

The **cosine is essentially 1**: both estimators point the same way, confirming
the baseline did not bias anything. And the **variance drops by ~18%**, lifting
the signal-to-noise ratio. Same direction, less noise — exactly what the theory
promised.

### When does a baseline help most?

The gain depends entirely on how much $v_\pi(s)$ **varies across states**. The
baseline removes the part of the return explained by "which state am I in";
if every state has a similar value, there is nothing to remove.

The uniform policy has values spanning `-4.74` to `+5.24`, so subtracting them
helps. Let us check a policy whose state values are nearly equal.

In [ ]:
theta_mid = theta_rf * 0.3          # a partly-trained, still-stochastic policy
v_mid = evaluate_exact(policy_probs(theta_mid))

rng = np.random.default_rng(4)
gm_p = gradient_samples(theta_mid, 4_000, rng)
rng = np.random.default_rng(4)
gm_b = gradient_samples(theta_mid, 4_000, rng, baseline=v_mid)

vp2, vb2 = gm_p.var(axis=0).mean(), gm_b.var(axis=0).mean()
spread_u = v_uniform[:4].std()
spread_m = v_mid[:4].std()

print(f"{'policy':<20}{'sd of v_pi(s)':>15}{'variance change':>18}")
print(f"{'uniform':<20}{spread_u:>15.3f}{100 * (1 - var_b / var_p):>17.1f}%")
print(f"{'partly trained':<20}{spread_m:>15.3f}{100 * (1 - vb2 / vp2):>17.1f}%")
print(f"\npartly-trained v_pi(s) = {np.round(v_mid[:4], 2)}")

The partly-trained policy's state values are bunched together, so its baseline
subtracts nearly the same constant everywhere — and the variance reduction
vanishes or even goes slightly negative (the baseline is estimated from the same
finite sample, so it can add a little noise of its own).

The lesson generalises: **a baseline removes between-state variance, not
within-state variance.** In this MDP most of the noise comes from the terminal
95/5 gamble — randomness *within* a state that no state-dependent baseline can
touch. To cut that you need a different tool, such as an action-dependent
critic or bootstrapping.

So a value baseline is worth having, but it is not magic, and the honest headline
is "modest, situation-dependent improvement" rather than "variance solved".

## REINFORCE with a learned critic

Using the true $v_\pi$ as a baseline was cheating; it came from the model. In
practice you **learn** the baseline alongside the policy. A learned value
function used this way is called a **critic**, and the pair is *actor-critic*:

- the **actor** $\pi_\theta$ chooses actions
- the **critic** $V_w$ estimates how good states are, supplying the baseline

The critic trains with the same incremental rule from notebook 03:

$$V(s) \leftarrow V(s) + \beta\,[G_t - V(s)]$$

Two things now learn at once, which is the standard actor-critic subtlety: the
critic is chasing a policy that keeps changing, so a *constant* step size $\beta$
is the right choice here — exactly the tracking behaviour notebook 03 flagged.

In [ ]:
def reinforce_critic(n_episodes, alpha, beta, rng, gamma=GAMMA, use_baseline=True):
    # REINFORCE with a learned value baseline. Returns (theta, V, curve, visits).
    theta = np.zeros((N_STATES, N_ACTIONS))
    V = np.zeros(N_STATES)
    visits = np.zeros(N_STATES)
    curve = np.zeros(n_episodes)

    for ep in range(n_episodes):
        traj = run_episode(theta, rng)
        Gs = returns_along(traj, gamma)
        grad = np.zeros_like(theta)

        for t, ((s, a, _), G) in enumerate(zip(traj, Gs)):
            advantage = (G - V[s]) if use_baseline else G
            grad += (gamma ** t) * advantage * grad_log_pi(theta, s, a)
            V[s] += beta * (G - V[s])            # critic update: new = old + step * error
            visits[s] += 1

        theta += alpha * grad
        curve[ep] = Gs[0]
    return theta, V, curve, visits


rng = np.random.default_rng(1)
theta_ac, V_ac, curve_ac, visits_ac = reinforce_critic(3_000, alpha=0.02, beta=0.05, rng=rng)

report(theta_ac, "actor-critic after 3,000 episodes")

print("\nlearned critic vs the true optimal values:")
print(f"  {'state':<18}{'V_critic':>10}{'v*':>10}{'error':>9}{'visits':>9}")
for s in NONTERMINAL:
    print(f"  {s.name:<18}{V_ac[s]:>10.3f}{V_STAR[s]:>10.3f}"
          f"{V_ac[s] - V_STAR[s]:>+9.3f}{int(visits_ac[s]):>9}")

The actor found an optimal policy, and the critic learned sensible values for
most states **without ever seeing the model**.

But one row is badly wrong, and it is worth understanding rather than glossing:
`SALINITY_CHECKED` is off by several units. The `visits` column explains it. The
actor converged on checking *fouling* first, so `SALINITY_CHECKED` is almost
never entered — it gets a small fraction of the updates the other states get,
and with a constant step size $\beta$ its estimate is dominated by a handful of
recent returns rather than a converged average.

This is the same coverage problem from notebook 03, now biting a critic instead
of an evaluator: **you cannot learn values for states your policy avoids.** It is
mostly harmless here — a bad baseline in a rarely-visited state barely affects
the gradient, because that state barely contributes to it — but it is exactly
the failure that becomes serious when a critic's errors feed back into the
policy, as in the bootstrapped methods PPO builds on.

Note too that the critic estimates $v_\pi$ for the *current* $\pi$, not $v_*$. As
the policy sharpens toward optimal the two converge, but they are not the same
object mid-training.

### Does the baseline actually speed up learning?

Lower gradient variance *should* mean faster, steadier learning. That is the
claim; let us test it across many seeds rather than trust it.

In [ ]:
def learning_summary(use_baseline, seeds=20, n_episodes=1_500, alpha=0.02, beta=0.05):
    finals, curves = [], []
    for sd in range(seeds):
        r = np.random.default_rng(100 + sd)
        th, _, c, _ = reinforce_critic(n_episodes, alpha, beta, r,
                                       use_baseline=use_baseline)
        finals.append(c[-200:].mean())
        curves.append(c)
    return np.array(finals), np.array(curves)


plain_finals, plain_curves = learning_summary(False)
base_finals, base_curves = learning_summary(True)

n = len(plain_finals)
print(f"final mean return over {n} seeds (last 200 episodes):")
print(f"  no baseline   : {plain_finals.mean():+.3f}  (sd across seeds {plain_finals.std():.3f})")
print(f"  with baseline : {base_finals.mean():+.3f}  (sd across seeds {base_finals.std():.3f})")
print(f"  optimal       : {V_STAR[State.NO_INFO]:+.3f}")

# Is the difference bigger than the seed-to-seed noise?
diff = base_finals.mean() - plain_finals.mean()
se = np.sqrt(plain_finals.var() / n + base_finals.var() / n)
print(f"\n  difference {diff:+.3f}  +/- {se:.3f} (standard error)"
      f"  -> {'significant' if abs(diff) > 2 * se else 'NOT significant'}")

print("\nmean return by training stage:")
print(f"  {'episodes':>9}{'no baseline':>14}{'with baseline':>15}")
for lo, hi in [(0, 200), (200, 500), (500, 900), (900, 1500)]:
    print(f"  {f'{lo}-{hi}':>9}{plain_curves[:, lo:hi].mean():>14.3f}"
          f"{base_curves[:, lo:hi].mean():>15.3f}")

Read the significance line before drawing any conclusion.

On this MDP the two are **statistically indistinguishable** — both reach near-
optimal behaviour at a similar rate. That is the honest result, and it follows
directly from the variance analysis above: most of the noise here comes from the
terminal 95/5 gamble, which is *within-state* randomness that no state-value
baseline can remove. An 18% cut in gradient variance is simply not enough to
show up over seed-to-seed noise on a 6-state problem that both versions solve
comfortably.

So why use baselines at all? Because the picture changes with scale. Long
episodes accumulate far more between-state variance for a baseline to remove;
large state spaces make each sample scarcer; and expensive environments make
sample efficiency the binding constraint. The technique is sound and standard —
this toy is just too forgiving to show it off.

Reporting "no significant difference" is the right outcome here. A tutorial that
manufactured a win by cherry-picking a seed would teach the method *and* a bad
habit.

## What we have, and what still hurts

REINFORCE optimises a policy directly from experience. A baseline or critic cuts
its variance without biasing it. But one problem remains, and it is the one PPO
was built for.

The gradient is only valid **at the current $\theta$**. Take too large a step and
the data you collected no longer describes the policy you now have, so the update
can make things worse — sometimes catastrophically, since a policy that collapses
onto a bad action stops collecting the data needed to escape.

The blunt fix is a tiny learning rate, which wastes every sample. Notebook 05
does better: it reuses each batch for **several** updates while explicitly
constraining how far the policy may move, via a probability ratio and clipping.

In [ ]:
# The failure mode, made concrete: too large a step size.
# Averaged over 10 seeds - a single seed is not enough to see this, as the
# earlier bias experiment in notebook 03 should have taught us by now.
print(f"{'alpha':>7}{'mean return':>13}{'sd':>8}{'worst seed':>12}{'optimal':>9}")
for alpha in [0.02, 0.1, 0.5, 1.0, 2.0, 5.0]:
    finals, opts = [], []
    for sd in range(10):
        r = np.random.default_rng(200 + sd)
        th, _, c, _ = reinforce_critic(800, alpha=alpha, beta=0.05, rng=r)
        finals.append(c[-200:].mean())
        opts.append(n_optimal(th))
    print(f"{alpha:>7}{np.mean(finals):>+13.3f}{np.std(finals):>8.3f}"
          f"{min(finals):>+12.2f}{np.mean(opts):>8.1f}/4")

There it is. At `alpha=0.02` learning is reliable: `+5.9` on average with a
standard deviation of `0.3`, and all four states end up optimal on every seed.
By `alpha=0.5` the mean has collapsed to around `+1` with a standard deviation
near `7` — and the worst seed finishes at about `-12`, meaning the policy locked
onto a *catastrophic* action and never recovered.

That last part is the crucial failure mode. A softmax policy that takes one
enormous step can drive an action's probability so low that it is essentially
never sampled again — and REINFORCE can only learn about actions it samples. The
policy destroys the very data it would need to recover.

Small steps are safe but slow; large steps are fast until they are fatal. There
is no single learning rate that is both, which is exactly the gap PPO closes.

| Notebook | Adds |
| --- | --- |
| 01 | states, actions, transitions, rewards, terminal states |
| 02 | Bellman equations, value iteration, policy extraction |
| 03 | Monte Carlo policy evaluation from sampled episodes |
| **04 — this one** | REINFORCE, baselines, and a learned critic |
| **05** | PPO's probability ratio and clipping |

Next: **`05_ppo_clipping.ipynb`**.